# GR00T Inference

This tutorial shows how to use the GR00T inference model to predict the actions from the observations, given a test dataset.

> **Note:** Select the Python kernel from the project virtual environment (`.venv`) before running. In Jupyter, go to **Kernel > Change Kernel** and select the `.venv` Python 3.10 environment.

In [1]:
import os
import torch
import gr00t

from gr00t.data.dataset.lerobot_episode_loader import LeRobotEpisodeLoader
from gr00t.data.dataset.sharded_single_step_dataset import extract_step_data
from gr00t.data.embodiment_tags import EmbodimentTag
from gr00t.policy.gr00t_policy import Gr00tPolicy

/pfss/mlde/workspaces/mlde_wsp_IAS_SAMMerge/VLA/duc/gr17_tta/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# change the following paths
MODEL_PATH = "/pfss/mlde/workspaces/mlde_wsp_IAS_SAMMerge/VLA/duc/gr17_tta/outputs/20260714_085102_baseline_merged_libero_1gpus_20000steps_100bs_10000ss/checkpoint-20000"

# REPO_PATH is the path of the pip install gr00t repo and one level up
REPO_PATH = os.path.dirname(os.path.dirname(gr00t.__file__))
DATASET_PATH = os.path.join(REPO_PATH, "/pfss/mlde/workspaces/mlde_wsp_IAS_SAMMerge/VLA/duc/gr17_tta/CP/merged_libero_mask_depth_noops_lerobot_10")
EMBODIMENT_TAG = "LIBERO_PANDA"

device = "cuda" if torch.cuda.is_available() else "cpu"

## Loading Pretrained Policy

Policy Model is loaded just like any other huggingface model.

There are 2 new concepts here in the GR00T model:
 - modality config: This defines the keys in the dictionary used by the model. (e.g. `action`, `state`, `annotation`, `video`)
 - modality_transform: A sequence of transform which are used during dataloading

In [3]:
policy = Gr00tPolicy(
    model_path=MODEL_PATH,
    embodiment_tag=EmbodimentTag.resolve(EMBODIMENT_TAG),
    device=device,
    strict=True,
)

# Print parameter count summary
total = sum(p.numel() for p in policy.model.parameters())
trainable = sum(p.numel() for p in policy.model.parameters() if p.requires_grad)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,} ({100*trainable/total:.1f}%)")

/pfss/mlde/workspaces/mlde_wsp_IAS_SAMMerge/VLA/duc/gr17_tta/.venv/lib/python3.10/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in Qwen3VLForConditionalGeneration is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", dtype=torch.float16)`
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in Qwen3VLModel is torch.float32. You should run training or inference

Total number of DiT parameters:  1091722240
Total number of SelfAttentionTransformer parameters:  201433088


Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  5.53it/s]


Total parameters: 3,144,016,000
Trainable parameters: 1,620,515,968 (51.5%)


## Loading dataset

First this requires user to check which embodiment tags are used to pretrained the `Gr00tPolicy` pretrained models.

In [4]:
import numpy as np

modality_config = policy.get_modality_config()

print(modality_config.keys())

for key, value in modality_config.items():
    if isinstance(value, np.ndarray):
        print(key, value.shape)
    else:
        print(key, value)


dict_keys(['video', 'state', 'action', 'language'])
video ModalityConfig(delta_indices=[0], modality_keys=['image', 'wrist_image'], sin_cos_embedding_keys=None, mean_std_embedding_keys=None, action_configs=None)
state ModalityConfig(delta_indices=[0], modality_keys=['x', 'y', 'z', 'roll', 'pitch', 'yaw', 'gripper'], sin_cos_embedding_keys=None, mean_std_embedding_keys=None, action_configs=None)
action ModalityConfig(delta_indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], modality_keys=['x', 'y', 'z', 'roll', 'pitch', 'yaw', 'gripper'], sin_cos_embedding_keys=None, mean_std_embedding_keys=None, action_configs=[ActionConfig(rep=<ActionRepresentation.ABSOLUTE: 'absolute'>, type=<ActionType.NON_EEF: 'non_eef'>, format=<ActionFormat.DEFAULT: 'default'>, state_key=None), ActionConfig(rep=<ActionRepresentation.ABSOLUTE: 'absolute'>, type=<ActionType.NON_EEF: 'non_eef'>, format=<ActionFormat.DEFAULT: 'default'>, state_key=None), ActionConfig(rep=<ActionRepresentation.ABSOLUTE: 'a

In [5]:
# Create the dataset
dataset = LeRobotEpisodeLoader(
    dataset_path=DATASET_PATH,
    modality_configs=modality_config,
)

In [6]:
# Validate that dataset contains all modality keys expected by the model.
# A mismatch here (e.g., from using a different model version) causes
# dimension errors later. See: https://github.com/NVIDIA/Isaac-GR00T/issues/305
import json as _json
_modality_path = os.path.join(DATASET_PATH, "meta", "modality.json")
with open(_modality_path) as _f:
    _dataset_modality = _json.load(_f)

for _mod in ["state", "action", "video"]:
    _dataset_keys = set(_dataset_modality.get(_mod, {}).keys())
    _model_keys = set(modality_config[_mod].modality_keys)
    _missing = _model_keys - _dataset_keys
    if _missing:
        print(f"ERROR: {_mod} — model requires keys not in dataset: {sorted(_missing)}")
        print(f"  Dataset keys: {sorted(_dataset_keys)}")
        print(f"  Model keys:   {sorted(_model_keys)}")
        raise ValueError(f"Dataset is missing required {_mod} keys: {sorted(_missing)}")
    _extra = _dataset_keys - _model_keys
    if _extra:
        print(f"{_mod}: OK ({len(_model_keys)} keys used, {len(_extra)} extra in dataset ignored)")
    else:
        print(f"{_mod}: OK ({len(_model_keys)} keys, exact match)")

state: OK (7 keys, exact match)
action: OK (7 keys, exact match)
video: OK (2 keys used, 2 extra in dataset ignored)


Let's print out a single data and visualize it

In [7]:
import numpy as np

episode_data = dataset[0]
step_data = extract_step_data(
    episode_data, step_index=0, modality_configs=modality_config, embodiment_tag=EmbodimentTag.resolve(EMBODIMENT_TAG), allow_padding=False
)

# print(step_data)

print("\n\n====================================")
print("Images:")
for img_key in step_data.images:
    print(" " * 4, img_key, f"{len(step_data.images[img_key])} x {step_data.images[img_key][0].shape}")

print("\nStates:")
for state_key in step_data.states:
    print(" " * 4, state_key, step_data.states[state_key].shape)

print("\nActions:")
for action_key in step_data.actions:
    print(" " * 4, action_key, step_data.actions[action_key].shape)

print("\nTask: ", step_data.text)




Images:
     image 1 x (256, 256, 3)
     wrist_image 1 x (256, 256, 3)

States:
     x (1, 1)
     y (1, 1)
     z (1, 1)
     roll (1, 1)
     pitch (1, 1)
     yaw (1, 1)
     gripper (1, 2)

Actions:
     x (16, 1)
     y (16, 1)
     z (16, 1)
     roll (16, 1)
     pitch (16, 1)
     yaw (16, 1)
     gripper (16, 1)

Task:  put both the alphabet soup and the cream cheese box in the basket


In [8]:
episode_data.keys()

Index(['language.annotation.human.action.task_description', 'state.x',
       'state.y', 'state.z', 'state.roll', 'state.pitch', 'state.yaw',
       'state.gripper', 'action.x', 'action.y', 'action.z', 'action.roll',
       'action.pitch', 'action.yaw', 'action.gripper', 'video.image',
       'video.wrist_image'],
      dtype='object')

In [16]:
episode_data['language.annotation.human.action.task_description'][0]

'put both the alphabet soup and the cream cheese box in the basket'

In [9]:
episode_data['state.x'].shape

(261,)

In [10]:
images = np.stack(episode_data['state.x'].to_numpy())
sam = torch.from_numpy(images)
sam.shape


torch.Size([261, 1])

In [11]:
sam.shape

torch.Size([261, 1])

Let's plot just the "right arm" state and action data and see how it looks like. Also show the images of the right hand state.

In [12]:
import matplotlib.pyplot as plt

episode_index = 0
max_steps = min(400, len(dataset[episode_index]))
joint_name = "right_arm"
image_key = "ego_view_bg_crop_pad_res256_freq20"

state_joints_across_time = []
gt_action_joints_across_time = []
images = []

sample_images = 6
episode_data = dataset[episode_index]
print(len(episode_data))

for step_count in range(max_steps):
    data_point = extract_step_data(
        episode_data, step_index=step_count, modality_configs=modality_config, embodiment_tag=EmbodimentTag.resolve(EMBODIMENT_TAG), allow_padding=False
    )
    state_joints = data_point.states[joint_name][0]
    gt_action_joints = data_point.actions[joint_name][0]

    state_joints_across_time.append(state_joints)
    gt_action_joints_across_time.append(gt_action_joints)

    # We can also get the image data
    if step_count % (max_steps // sample_images) == 0:
        image = data_point.images[image_key][0]
        images.append(image)

# Size is (max_steps, num_joints)
state_joints_across_time = np.array(state_joints_across_time)
gt_action_joints_across_time = np.array(gt_action_joints_across_time)


# Plot the joint angles across time
num_joints = state_joints_across_time.shape[1]
fig, axes = plt.subplots(nrows=num_joints, ncols=1, figsize=(8, 2*num_joints))

for i, ax in enumerate(axes):
    ax.plot(state_joints_across_time[:, i], label="state joints")
    ax.plot(gt_action_joints_across_time[:, i], label="gt action joints")
    ax.set_title(f"Joint {i}")
    ax.legend()

plt.tight_layout()
plt.show()


# Plot the images in a row
fig, axes = plt.subplots(nrows=1, ncols=sample_images, figsize=(16, 4))

for i, ax in enumerate(axes):
    ax.imshow(images[i])
    ax.axis("off")

261


KeyError: 'right_arm'

Now we can run the policy from the pretrained checkpoint.

In [13]:
observation = {
    "video": {k: np.stack(step_data.images[k])[None] for k in step_data.images},  # stack images and add batch dimension
    "state": {k: step_data.states[k][None] for k in step_data.states},  # add batch dimension
    "action": {k: step_data.actions[k][None] for k in step_data.actions},  # add batch dimension
    "language": {
        modality_config["language"].modality_keys[0]: [[step_data.text]],  # add time and batch dimension
    }
}
predicted_action, info = policy.get_action(observation)
for key, value in predicted_action.items():
    print(key, value.shape)

x (1, 16, 1)
y (1, 16, 1)
z (1, 16, 1)
roll (1, 16, 1)
pitch (1, 16, 1)
yaw (1, 16, 1)
gripper (1, 16, 1)


In [17]:
modality_config["language"].modality_keys[0]

'annotation.human.action.task_description'

In [14]:
observation['state']['x'].shape

(1, 1, 1)

In [ ]:
observation['video']['image'].shape

(1, 1, 256, 256, 3)

For more details on the policy (e.g. expected input and output), please refer to the [policy documentation](policy.md).